In [3]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.documents import Document

# 1.知识加载 - 将PDF文件使用文件加载器，加载成内存文档
loader = PyPDFLoader(
    # 支持绝对路径，也支持相对路径
    file_path="resources/贵州茅台研报.pdf",
    mode="single"   # page:代表按页加载文件
)
docs:list[Document] = loader.load()
print(f"""
    文档数据：{len(docs)},
    第一个文档的内容：{docs[0].page_content}
""")


# -------------------

from langchain_text_splitters import CharacterTextSplitter
# 2.知识切分 - 将加载的文件内容，切分为知识片段
splitter = CharacterTextSplitter(
    separator="\n",
    chunk_size=800,
    chunk_overlap=100
)
chunks:list[Document] = splitter.split_documents(docs)
print(f"""
    切分后的片段数据：{len(chunks)},
    切分后的第一个文本：{chunks[0].page_content}
""")

# -----------------------

from langchain_core.vectorstores import InMemoryVectorStore
from langchain_ollama import OllamaEmbeddings
# 3.向量化 - 不用我们自行处理，只需要将模型交给向量数据库，会自动的进行向量化
embeddings = OllamaEmbeddings(
    model="qwen3-embedding:0.6b",
    dimensions=1024,
    base_url="http://localhost:11434"
)
# 4.将数据存储到向量化数据库中
vectorstore = InMemoryVectorStore.from_documents(chunks, embeddings)


    切分后的片段数据：13,
    切分后的第一个文本：请务必阅读正文之后的免责声明及其项下所有内容
证券研究报告 | 2026年04月25日
优于大市贵州茅台（600519.SH）
2026Q1 收入稳健增长，市场化改革成效逐步显现
核心观点 公司研究·财报点评
食品饮料·白酒Ⅱ
证券分析师：张向伟 证券分析师：张未艾
021-61761031
zhangxiangwei@guosen.com.cn zhangweiai@guosen.com.cn
S0980523090001 S0980525070005
基础数据
投资评级 优于大市(维持)
合理估值
收盘价 1458.49 元
总市值/流通市值 1826424/1826424 百万元
52周最高价/最低价 1645.00/1322.01 元
近3 个月日均成交额 6551.18 百万元
市场走势
资料来源：Wind、国信证券经济研究所整理
相关研究报告
《贵州茅台（600519.SH）-2025 年释放经营压力，2026 年改革
破局》 ——2026-04-18
《贵州茅台（600519.SH）-革故鼎新，与时偕行，再论茅台的护
城河与增长潜力》 ——2026-03-19
《贵州茅台（600519.SH）-全面推进市场化转型，解决供需适配
问题》 ——2026-01-05
《贵州茅台（600519.SH）-2025Q3 收入同比持平，经营理性务实、
行稳致远》 ——2025-10-30
《贵州茅台（600519.SH）-2025Q2 收入同比增长 9.2%，公司根
据市场调整产品结构》 ——2025-08-13
2026Q1收入同比+6.3%，市场化改革下，经营韧性显现。2026Q1公司实现营业
总收入 547 亿元/同比+6.3%，归母净利润 272 亿元/同比+1.5%。结合
25Q4+26Q1看，收入同比-6.5%/净利润同比-14.0%/收入+合同负债环比变动



In [4]:
from app.core.config import settings
from langchain.chat_models import init_chat_model

# query = "茅台的收盘价是多少"
query = "茅台2025年的市盈率和市净率是多少"

# 1.相似度检索
doc_list = vectorstore.similarity_search(query)

# 2.组装提示词
for doc in doc_list:
    query += "\n" + doc.page_content

# 3.准备模型
model = init_chat_model(
    model="deepseek-flash",
    api_key=settings.llm.api_key
)

# 4.问答
ai_message = model.invoke(query)
ai_message.pretty_print()

================================== Ai Message ==================================

根据你提供的国信证券经济研究所预测资料：

- **贵州茅台2025年市盈率（PE）**：**22.2倍**
- **贵州茅台2025年市净率（PB）**：**7.47倍**

对应表中“2025A/E”或“2025”列数据。
